In [1]:
print("hello")

hello


# モデルの簡単な推論

In [ ]:
import torch
from transformers import AutoTokenizer
from trl import AutoModelForCausalLMWithValueHead
from peft import PeftModel

# 元モデル + LoRA 重み
base_model = "Qwen/Qwen2.5-3B-Instruct"
adapter = "creabridge_lora_ckpt"  # 学習保存したフォルダ

tokenizer = AutoTokenizer.from_pretrained(base_model, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# 元モデルをロード
model = AutoModelForCausalLMWithValueHead.from_pretrained(
    base_model,
    device_map="cuda",
    torch_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
)

# LoRA を適用
model = PeftModel.from_pretrained(model, adapter)
model = model.eval()

def generate(prompt):
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=200,
            temperature=0.6,
            top_p=0.9
        )
    return tokenizer.decode(out[0], skip_special_tokens=True)

# 例: 推論
print(generate("Suggest a novel interdisciplinary research idea about AI and biology."))

# モデルの性能評価

In [ ]:
import os, json, math
from typing import List, Tuple
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from trl import AutoModelForCausalLMWithValueHead
from peft import PeftModel

# ====== 設定（必要に応じて変更） ======
BASE_MODEL = "Qwen/Qwen2.5-3B-Instruct"             # 学習前モデル
ADAPTER_DIR = "creabridge_lora_ckpt"                # 学習後LoRAの保存先
SEEDS_PATH = "CORY_withRAG/data/research_seeds.fixed.jsonl"
NUM_SAMPLES = 50                                    # 評価に使うサンプル数

POLICY_DEVICE = "cuda:0"
IRM_MODEL_DIR = "ayarnte/Idea_Reward_Model"
IRM_DEVICE = "cuda:1"
IRM_MAX_LEN = 512

MAX_NEW_TOKENS = 96
TEMPERATURE = 0.2
TOP_P = 0.9

# ====== IRM スコアラー ======
class IRMScorer:
    def __init__(self, model_dir, max_len=512, device="cuda:1"):
        self.device = torch.device(device)
        self.tok = AutoTokenizer.from_pretrained(model_dir, use_fast=True)
        self.model = AutoModelForSequenceClassification.from_pretrained(model_dir).to(self.device).eval()
        self.max_len = max_len

    @torch.no_grad()
    def score(self, title, abstract):
        text = f"Title: {title}\nAbstract: {abstract}"
        enc = self.tok(text, truncation=True, max_length=self.max_len, return_tensors="pt")
        enc = {k: v.to(self.device) for k, v in enc.items()}
        out = self.model(**enc)
        logits = out.logits.squeeze(0)
        if logits.ndim <= 1:
            val = float(logits.item())
            return 1 / (1 + math.exp(-val))
        prob = torch.softmax(logits, dim=-1)
        return float(prob.max().item())

# ====== データ取得 ======
def load_tasks(path, n):
    tasks=[]
    with open(path,"r",encoding="utf-8") as f:
        for line in f:
            if len(tasks)>=n: break
            try:
                obj=json.loads(line)
                tasks.append(obj.get("prompt", obj.get("task", str(obj))))
            except:
                tasks.append(line.strip())
    return tasks

def parse_title_abstract(text):
    lines=[ln.strip() for ln in text.splitlines() if ln.strip()]
    title = lines[0] if lines else "Untitled"
    abstract = " ".join(lines[1:]) if len(lines) > 1 else ""
    return title[:200], abstract[:3000]

def choose_dtype():
    return torch.bfloat16 if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else torch.float16

@torch.no_grad()
def gen_batch(model, tok, prompts, device):
    enc = tok(prompts, return_tensors="pt", padding=True, truncation=True, max_length=512)
    enc = {k:v.to(device) for k,v in enc.items()}
    out = model.generate(
        **enc, do_sample=(TEMPERATURE>0), temperature=TEMPERATURE,
        top_p=TOP_P, max_new_tokens=MAX_NEW_TOKENS,
        pad_token_id=tok.eos_token_id, eos_token_id=tok.eos_token_id
    )
    texts = tok.batch_decode(out, skip_special_tokens=True)
    return [t.strip() for t in texts]

def eval_model(adapter_dir_or_none):
    tok = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=True)
    if tok.pad_token is None: tok.pad_token = tok.eos_token

    model = AutoModelForCausalLMWithValueHead.from_pretrained(
        BASE_MODEL, torch_dtype=choose_dtype(),
        device_map={"": POLICY_DEVICE},
    )
    if adapter_dir_or_none:
        model = PeftModel.from_pretrained(model, adapter_dir_or_none)
    model.eval()

    outputs = gen_batch(model, tok, [f"Task: {t}\nWrite Title and Abstract." for t in tasks], POLICY_DEVICE)

    scores=[]
    for txt in outputs:
        title, abs_ = parse_title_abstract(txt)
        s=irm.score(title, abs_)
        scores.append(s)

    del model
    torch.cuda.empty_cache()

    return float(torch.tensor(scores).mean()), float(torch.tensor(scores).std()), len(scores)

# ====== 実行 ======
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

tasks = load_tasks(SEEDS_PATH, NUM_SAMPLES)
irm = IRMScorer(IRM_MODEL_DIR, max_len=IRM_MAX_LEN, device=IRM_DEVICE)

print("=== BEFORE (base model) ===")
b_avg, b_std, b_n = eval_model(None)
print(f"avg={b_avg:.4f}, std={b_std:.4f}, n={b_n}")

print("=== AFTER (LoRA applied) ===")
a_avg, a_std, a_n = eval_model(ADAPTER_DIR)
print(f"avg={a_avg:.4f}, std={a_std:.4f}, n={a_n}")

print(f"\n=== LIFT (after - before) = {a_avg - b_avg:+.4f}")



'''
出力例
=== BEFORE (base model) ===
avg=0.4621, std=0.0512, n=50
=== AFTER (LoRA applied) ===
avg=0.5318, std=0.0489, n=50

=== LIFT (after - before) = +0.0697
'''

# モデルの創造性スコア評価

In [ ]:
# --- CREA-Bridge: 創造性スコアのワンセル評価パイプライン ---
# 必要パッケージ:
# pip install sentence-transformers

import os, json, math, random, time, re, gc
import numpy as np
import pandas as pd
from pathlib import Path
from typing import List, Dict, Any, Tuple

import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from trl import AutoModelForCausalLMWithValueHead
from peft import PeftModel
from sentence_transformers import SentenceTransformer, util

# ========================= ユーザ設定（必要なら編集） =========================
BASE_MODEL = "Qwen/Qwen2.5-3B-Instruct"
ADAPTER_DIR = "creabridge_lora_ckpt"                    # 学習済みLoRAの保存先
SEEDS_PATH = "CORY_withRAG/data/research_seeds.fixed.jsonl"  # 既存研究/シード
IRM_MODEL_DIR = "ayarnte/Idea_Reward_Model"            # IRMモデル（分類or確率出力）
SAVE_CSV = "creativity_eval_results.csv"
SAVE_SUMMARY = "creativity_eval_summary.json"

N_PROMPTS = 50          # 評価するプロンプト数（負荷に応じて調整）
K_CANDIDATES = 3        # 1プロンプトあたり生成案の数
MAX_NEW_TOKENS = 200
TEMPERATURE = 0.6
TOP_P = 0.9
SEED = 42

# Creativity スコアの重み
W_USEFUL = 0.5
W_SPECIF = 0.3
W_COHER  = 0.2

# 可能なら policy=GPU0, IRM=GPU1
POLICY_DEVICE = "cuda:0" if torch.cuda.is_available() and torch.cuda.device_count()>=1 else "cpu"
IRM_DEVICE    = "cuda:1" if torch.cuda.is_available() and torch.cuda.device_count()>=2 else ("cuda:0" if torch.cuda.is_available() else "cpu")

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# ========================= ユーティリティ =========================
def safe_json_loads(line: str):
    try:
        return json.loads(line)
    except Exception:
        return None

def extract_title_desc(obj: Any) -> Tuple[str, str]:
    """シード行から title/desc を抽出。無ければ 'prompt' やまとめ文字列から生成。"""
    title, desc = "", ""
    if isinstance(obj, dict):
        title = str(obj.get("title") or obj.get("Topic") or obj.get("problem") or obj.get("task") or obj.get("prompt") or "")[:200]
        desc  = str(obj.get("abstract") or obj.get("desc") or obj.get("context") or obj.get("Prompt") or obj.get("text") or "")[:4000]
        if not title and not desc:
            title = str(obj)[:200]; desc = str(obj)[:4000]
    else:
        s = str(obj); title = s[:100]; desc = s[:4000]
    return title, desc

def load_seed_prompts(path: str, max_n: int = None) -> List[str]:
    prompts = []
    with open(path, "r", encoding="utf-8") as f:
        for i, line in enumerate(f):
            if max_n is not None and len(prompts) >= max_n:
                break
            obj = safe_json_loads(line)
            if obj is None:
                s = line.strip()
                if s: prompts.append(s)
                continue
            # prompt優先、なければ problem/topic/abstract などを連結
            p = obj.get("prompt") or obj.get("task") or obj.get("problem") or obj.get("topic")
            if not p:
                t, d = extract_title_desc(obj)
                p = (t + "\n" + d).strip()
            if p:
                prompts.append(str(p))
    # シャッフル安定
    rng = random.Random(SEED); rng.shuffle(prompts)
    return prompts[:max_n] if max_n else prompts

def distinct_n(text: str, n: int = 2) -> float:
    toks = re.findall(r"\w+", text.lower())
    if len(toks) < n: return 0.0
    grams = [" ".join(toks[i:i+n]) for i in range(len(toks)-n+1)]
    return len(set(grams)) / max(1, len(grams))

# ========================= 埋め込みモデル（新規性・多様性） =========================
def load_embedder():
    # Sci分野なら SPECTER2 を優先、失敗時 mpnet にフォールバック
    candidates = ["allenai/specter2_base", "sentence-transformers/all-mpnet-base-v2"]
    last_e = None
    for name in candidates:
        try:
            return SentenceTransformer(name)
        except Exception as e:
            last_e = e
            continue
    raise RuntimeError(f"Embedding model load failed: {last_e}")

def build_corpus_embeddings(embedder, seeds_path: str, max_n: int = 2000):
    # 既存研究/シードから title+desc を集め埋め込み
    texts = []
    with open(seeds_path, "r", encoding="utf-8") as f:
        for i, line in enumerate(f):
            if len(texts) >= max_n: break
            obj = safe_json_loads(line)
            if obj is None:
                s = line.strip()
                if s: texts.append(s)
                continue
            t, d = extract_title_desc(obj)
            if not t and not d:
                # 最低限 prompt でも入れる
                p = obj.get("prompt") or obj.get("task") or obj.get("problem") or ""
                texts.append(str(p))
            else:
                texts.append((t + "\n" + d).strip())
    if len(texts) == 0:
        texts = ["Large Language Model research idea about safety and efficiency."]
    embs = embedder.encode(texts, convert_to_tensor=True, normalize_embeddings=True, batch_size=64, show_progress_bar=False)
    return embs

def novelty_score(embedder, idea_text: str, corpus_embs) -> float:
    emb = embedder.encode([idea_text], convert_to_tensor=True, normalize_embeddings=True)
    sim = util.cos_sim(emb, corpus_embs).max().item()
    return float(max(0.0, min(1.0, 1.0 - sim)))  # 高いほど新規

def diversity_scores(embedder, ideas: List[str]) -> Dict[str, float]:
    if len(ideas) <= 1:
        return {"pair_avg_cos_dist": 0.0, "distinct2_avg": distinct_n(ideas[0],2) if ideas else 0.0}
    embs = embedder.encode(ideas, convert_to_tensor=True, normalize_embeddings=True)
    # すべてのペア距離
    dists = []
    for i in range(len(ideas)):
        for j in range(i+1, len(ideas)):
            s = util.cos_sim(embs[i], embs[j]).item()
            dists.append(1.0 - s)
    return {
        "pair_avg_cos_dist": float(np.mean(dists)),
        "distinct2_avg": float(np.mean([distinct_n(x,2) for x in ideas]))
    }

# ========================= IRM（有用性） =========================
class IRMScorer:
    def __init__(self, model_dir: str, device: str = "cpu", max_len: int = 512):
        self.device = device
        self.tok = AutoTokenizer.from_pretrained(model_dir, use_fast=True)
        self.model = AutoModelForSequenceClassification.from_pretrained(model_dir).to(self.device).eval()
        self.max_len = max_len
    @torch.inference_mode()
    def score(self, title: str, desc: str) -> float:
        text = f"Title: {title}\nAbstract: {desc}"
        enc = self.tok(text, truncation=True, max_length=self.max_len, return_tensors="pt")
        enc = {k: v.to(self.device) for k, v in enc.items()}
        out = self.model(**enc).logits.squeeze(0)
        if out.ndim==0 or out.numel()==1:
            # バイナリ・スカラー出力想定
            val = float(out.item())
            return 1.0 / (1.0 + math.exp(-val))
        else:
            prob = torch.softmax(out, dim=-1)
            return float(prob.max().item())

# ========================= 明確性/具体性・一貫性スコア =========================
SPEC_CHECK_PROMPT = """You are a strict reviewer. For the given research idea, answer 4 binary flags:
(1) problem definition present?
(2) method outline present?
(3) evaluation plan present?
(4) risks/limitations mentioned?
Return exactly four digits (0/1) separated by a single space, like: "1 0 1 1".
Idea:
"""

COHER_CHECK_PROMPT = """Judge internal coherence of the following idea on a 0-1 scale (0=contradictory or vague, 1=coherent and logically consistent). Answer with a single number between 0 and 1.
Idea:
"""

def parse_4bits(text: str) -> List[int]:
    bits = re.findall(r"[01]", text)
    bits = [int(b) for b in bits[:4]]
    while len(bits)<4: bits.append(0)
    return bits

@torch.inference_mode()
def run_critic_bits(model, tokenizer, idea: str, device: str) -> Tuple[float,float]:
    # 明確性（4項目充足率）と一貫性（0-1）を、同一policyで簡易推定（外部依存なし）
    # ※ 評価リーク懸念があれば、別の小型モデルを使うように切替可能
    inp = tokenizer(SPEC_CHECK_PROMPT + idea, return_tensors="pt").to(device)
    out = model.generate(**inp, max_new_tokens=64, temperature=0.0, top_p=1.0, pad_token_id=tokenizer.eos_token_id)
    txt = tokenizer.decode(out[0], skip_special_tokens=True)
    bits = parse_4bits(txt)
    specificity = sum(bits)/4.0

    inp2 = tokenizer(COHER_CHECK_PROMPT + idea, return_tensors="pt").to(device)
    out2 = model.generate(**inp2, max_new_tokens=32, temperature=0.0, top_p=1.0, pad_token_id=tokenizer.eos_token_id)
    txt2 = tokenizer.decode(out2[0], skip_special_tokens=True)
    m = re.search(r"([01](?:\.\d+)?)", txt2)
    coherence = float(m.group(1)) if m else 0.5
    coherence = max(0.0, min(1.0, coherence))
    return specificity, coherence

# ========================= モデル（Baseline/LoRA） =========================
def load_policy(base_model: str, adapter_dir: str=None, device: str="cuda"):
    tok = AutoTokenizer.from_pretrained(base_model, use_fast=True)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    policy = AutoModelForCausalLMWithValueHead.from_pretrained(
        base_model,
        device_map=device if device!="cpu" else None,
        torch_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else (torch.float16 if torch.cuda.is_available() else torch.float32),
    )
    if adapter_dir:
        policy = PeftModel.from_pretrained(policy, adapter_dir)
    policy = policy.eval()
    return policy, tok

@torch.inference_mode()
def generate_many(model, tokenizer, prompt: str, k: int, max_new_tokens=200, temperature=0.6, top_p=0.9, device="cuda"):
    out_texts = []
    enc = tokenizer(prompt, return_tensors="pt").to(device)
    for _ in range(k):
        gen = model.generate(
            **enc,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            top_p=top_p,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
        txt = tokenizer.decode(gen[0], skip_special_tokens=True)
        # 入力プロンプト部分を可能なら削除
        if txt.startswith(prompt): txt = txt[len(prompt):]
        out_texts.append(txt.strip())
    return out_texts

def split_title_desc(text: str) -> Tuple[str,str]:
    lines = [l.strip() for l in text.splitlines() if l.strip()]
    title = lines[0][:200] if lines else "Untitled"
    desc  = text[:4000]
    return title, desc

def creativity_score(novelty, usefulness, specificity, coherence, w_use=W_USEFUL, w_sp=W_SPECIF, w_co=W_COHER):
    return float(novelty * (w_use*usefulness + w_sp*specificity + w_co*coherence))

# ========================= 実行フロー =========================
def main():
    os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
    print(f"[DEVICES] policy={POLICY_DEVICE}, irm={IRM_DEVICE}")

    # 1) シード読込（評価プロンプト群） & 埋め込み準備
    prompts = load_seed_prompts(SEEDS_PATH, N_PROMPTS)
    assert len(prompts)>0, "評価用プロンプトが空です"
    embedder = load_embedder()
    corpus_embs = build_corpus_embeddings(embedder, SEEDS_PATH, max_n=2000)

    # 2) IRM
    irm = IRMScorer(IRM_MODEL_DIR, device=IRM_DEVICE, max_len=512)

    # 3) モデル2系統（Baseline / LoRA適用）
    base_model, base_tok = load_policy(BASE_MODEL, adapter_dir=None, device=POLICY_DEVICE)
    lora_model, lora_tok = load_policy(BASE_MODEL, adapter_dir=ADAPTER_DIR, device=POLICY_DEVICE)

    # 4) 評価ループ
    rows = []
    for pi, prompt in enumerate(prompts):
        for cond_name, model, tok in [
            ("baseline", base_model, base_tok),
            ("creabridge_lora", lora_model, lora_tok),
        ]:
            ideas = generate_many(model, tok, prompt, k=K_CANDIDATES,
                                  max_new_tokens=MAX_NEW_TOKENS,
                                  temperature=TEMPERATURE, top_p=TOP_P,
                                  device=POLICY_DEVICE)
            # 多様性（同プロンプト内）
            div = diversity_scores(embedder, ideas)
            for ci, idea in enumerate(ideas):
                title, desc = split_title_desc(idea)
                nov = novelty_score(embedder, idea, corpus_embs)
                use = irm.score(title, desc)
                specif, coher = run_critic_bits(model, tok, idea, POLICY_DEVICE)
                creat = creativity_score(nov, use, specif, coher)

                rows.append({
                    "prompt_id": pi,
                    "condition": cond_name,
                    "candidate_id": ci,
                    "novelty": nov,
                    "usefulness_irm": use,
                    "specificity": specif,
                    "coherence": coher,
                    "div_pair_avg_cos_dist": div["pair_avg_cos_dist"],
                    "div_distinct2_avg": div["distinct2_avg"],
                    "creativity": creat,
                    "prompt": prompt[:2000],
                    "idea": idea[:5000],
                    "title": title[:200],
                })
        print(f"[{pi+1}/{len(prompts)}] done.")

    df = pd.DataFrame(rows)
    df.to_csv(SAVE_CSV, index=False, encoding="utf-8")
    print(f"[SAVE] {SAVE_CSV} に明細を保存")

    # 5) 集計（勝率/平均差）
    def agg_stats(metric: str):
        a = df[df["condition"]=="baseline"].groupby("prompt_id")[metric].mean()
        b = df[df["condition"]=="creabridge_lora"].groupby("prompt_id")[metric].mean()
        common = sorted(set(a.index)&set(b.index))
        a = a.loc[common]; b = b.loc[common]
        wins = (b > a).mean()  # LoRAが上回った割合
        diff = (b - a).mean()
        return {"win_rate": float(wins), "mean_diff": float(diff), "n": len(common)}

    summary = {
        "creativity": agg_stats("creativity"),
        "novelty": agg_stats("novelty"),
        "usefulness_irm": agg_stats("usefulness_irm"),
        "specificity": agg_stats("specificity"),
        "coherence": agg_stats("coherence"),
        "div_pair_avg_cos_dist": agg_stats("div_pair_avg_cos_dist"),
        "div_distinct2_avg": agg_stats("div_distinct2_avg"),
        "config": {
            "N_PROMPTS": N_PROMPTS, "K_CANDIDATES": K_CANDIDATES,
            "weights": {"use": W_USEFUL, "specificity": W_SPECIF, "coherence": W_COHER},
            "gen": {"max_new_tokens": MAX_NEW_TOKENS, "temperature": TEMPERATURE, "top_p": TOP_P},
            "devices": {"policy": POLICY_DEVICE, "irm": IRM_DEVICE},
            "embedder": str(embedder),
            "base_model": BASE_MODEL, "adapter": ADAPTER_DIR,
        }
    }

    with open(SAVE_SUMMARY, "w", encoding="utf-8") as f:
        json.dump(summary, f, ensure_ascii=False, indent=2)
    print(json.dumps(summary, ensure_ascii=False, indent=2))

    # メモリ掃除（長期実行時）
    del embedder, corpus_embs, base_model, lora_model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

main()